# Point Predictions
This notebook uses some of the metadata that was provided with the reviews to try to predict the variety

This metadata includes:
- country
- points(score)
- price
- province

Each of these models was trained and tested on the metadata:
- SVM
- K Neighbors
- Decision Tree
- Random Forest
- Hist Gradient Boost
- Gradient Boost

## Filter Data

In [2]:
import pandas as pd

df = pd.read_csv('Wine Reviews Dataset/winemag-data-130k-v2.csv')

varieties = [
    'Pinot Noir',
    'Chardonnay',
    'Cabernet Sauvignon',
    'Red Blend',
    'Bordeaux-style Red Blend',
    'Riesling',
    'Sauvignon Blanc',
    'Syrah',
    'Rosé',
    'Merlot'
]

# Keep only the rows with the varieties listed above
df = df[df['variety'].isin(varieties)]

# Replace the wine names with [WINE] so the model doesn't just learn the names of the wines
df['description'] = df['description'].replace(varieties, '[WINE]')

# Discard columns we won't be using
df = df.drop(columns=['Unnamed: 0', 'description', 'designation', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'winery'])

# Drop rows with NaN values
print(len(df))
df.dropna(subset=['points', 'price', 'variety'], inplace=True)
print(len(df))

df[0:10]

71322
67234


,country,points,price,province,variety
3,US,87,13.0,Michigan,Riesling
4,US,87,65.0,Oregon,Pinot Noir
10,US,87,19.0,California,Cabernet Sauvignon
12,US,87,34.0,California,Cabernet Sauvignon
14,US,87,12.0,California,Chardonnay
15,Germany,87,24.0,Mosel,Riesling
20,US,87,23.0,Virginia,Red Blend
21,US,87,20.0,Oregon,Pinot Noir
23,US,87,22.0,California,Merlot
25,US,87,69.0,California,Pinot Noir


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df['variety'])

X = df.drop(columns=['variety'])
# Normalize points and price
X['points'] = X['points'] / 100
X['price'] = X['price'] / X['price'].max()
# Convert categorical features to one-hot encoding
X = pd.get_dummies(X, columns=['country', 'province'])

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

## SVM

In [3]:
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, precision_score, recall_score

# Create and train the SVM model
svm = LinearSVC(max_iter=2000, dual='auto', C=15.0)
svm.fit(X_train, y_train)

# Predict the variety for the test set
y_pred = svm.predict(X_test)

# Decode the predicted variety
predicted_variety = le.inverse_transform(y_pred)

# Print the predicted variety
print(predicted_variety)

# Print the accuracy scores
print(f"{'Training accuracy: ':<20}{svm.score(X_train, y_train)}")
print(f"{'Testing accuracy: ':<20}{svm.score(X_test, y_test)}")

print(f"{'F1:':<20}{f1_score(y_test, y_pred, average='weighted')}")
print(f"{'Precision:':<20}{precision_score(y_test, y_pred, average='weighted')}")
print(f"{'Recall:':<20}{recall_score(y_test, y_pred, average='weighted')}")

['Chardonnay' 'Syrah' 'Riesling' ... 'Sauvignon Blanc' 'Red Blend'
 'Chardonnay']
Training accuracy:  0.480004462044732
Testing accuracy:   0.4759425894251506
F1:                 0.46531568471968454
Precision:          0.49505147237304103
Recall:             0.4759425894251506


## K Neighbors

In [5]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

# Create and train the KNN model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Predict the variety for the test set
y_pred = knn.predict(X_test)

# Decode the predicted variety
predicted_variety = le.inverse_transform(y_pred)

# Print the predicted variety
print(predicted_variety)

# Print the accuracy scores
print(f"{'Training accuracy: ':<20}{knn.score(X_train, y_train)}")
print(f"{'Testing accuracy: ':<20}{knn.score(X_test, y_test)}")

print(f"{'F1:':<20}{f1_score(y_test, y_pred, average='weighted')}")
print(f"{'Precision:':<20}{precision_score(y_test, y_pred, average='weighted')}")
print(f"{'Recall:':<20}{recall_score(y_test, y_pred, average='weighted')}")

['Chardonnay' 'Syrah' 'Riesling' ... 'Sauvignon Blanc' 'Red Blend'
 'Merlot']
Training accuracy:  0.5327681410006135
Testing accuracy:   0.46805979028779654
F1:                 0.46453765815152936
Precision:          0.4720653211423208
Recall:             0.46805979028779654


## Decision Tree

In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

# Create and train the Random Forest model
decisiontree = DecisionTreeClassifier(max_depth=20, random_state=2)
decisiontree.fit(X_train, y_train)

# Predict the variety for the test set
y_pred = decisiontree.predict(X_test)

# Decode the predicted variety
predicted_variety = le.inverse_transform(y_pred)

# Print the predicted variety
print(predicted_variety)

# Print the accuracy scores
print(f"{'Training accuracy: ':<20}{decisiontree.score(X_train, y_train)}")
print(f"{'Testing accuracy: ':<20}{decisiontree.score(X_test, y_test)}")

print(f"{'F1:':<20}{f1_score(y_test, y_pred, average='weighted')}")
print(f"{'Precision:':<20}{precision_score(y_test, y_pred, average='weighted')}")
print(f"{'Recall:':<20}{recall_score(y_test, y_pred, average='weighted')}")

['Chardonnay' 'Syrah' 'Riesling' ... 'Sauvignon Blanc' 'Red Blend'
 'Chardonnay']
Training accuracy:  0.5389034525071114
Testing accuracy:   0.4780992042834833
F1:                 0.4731857812668736
Precision:          0.5017500227999678
Recall:             0.4780992042834833


## Random Forest

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

# Create and train the Random Forest model
randomforest = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=2)
randomforest.fit(X_train, y_train)

# Predict the variety for the test set
y_pred = randomforest.predict(X_test)

# Decode the predicted variety
predicted_variety = le.inverse_transform(y_pred)

# Print the predicted variety
print(predicted_variety)

# Print the accuracy scores
print(f"{'Training accuracy: ':<20}{randomforest.score(X_train, y_train)}")
print(f"{'Testing accuracy: ':<20}{randomforest.score(X_test, y_test)}")

print(f"{'F1:':<20}{f1_score(y_test, y_pred, average='weighted')}")
print(f"{'Precision:':<20}{precision_score(y_test, y_pred, average='weighted')}")
print(f"{'Recall:':<20}{recall_score(y_test, y_pred, average='weighted')}")

['Chardonnay' 'Syrah' 'Riesling' ... 'Sauvignon Blanc' 'Red Blend'
 'Chardonnay']
Training accuracy:  0.582873184970346
Testing accuracy:   0.5056890012642224
F1:                 0.49873530746297756
Precision:          0.5256434946074491
Recall:             0.5056890012642224


## Hist Gradient Boosting
Runs much faster than Gradient Boosting but with worse results

In [15]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

# Create and train the Random Forest model
histgradientboosting = HistGradientBoostingClassifier(random_state=2)
# gradientboosting = Grad
# ntBoostingClassifier(random_state=2)

histgradientboosting.fit(X_train, y_train)

# Predict the variety for the test set
y_pred = histgradientboosting.predict(X_test)

# Decode the predicted variety
predicted_variety = le.inverse_transform(y_pred)

# Print the predicted variety
print(predicted_variety)

# Print the accuracy scores
print(f"{'Training accuracy: ':<20}{histgradientboosting.score(X_train, y_train)}")
print(f"{'Testing accuracy: ':<20}{histgradientboosting.score(X_test, y_test)}")

print(f"{'F1:':<20}{f1_score(y_test, y_pred, average='weighted')}")
print(f"{'Precision:':<20}{precision_score(y_test, y_pred, average='weighted')}")
print(f"{'Recall:':<20}{recall_score(y_test, y_pred, average='weighted')}")

['Chardonnay' 'Syrah' 'Riesling' ... 'Sauvignon Blanc' 'Red Blend'
 'Chardonnay']
Training accuracy:  0.5272835443508654
Testing accuracy:   0.5056146352346248
F1:                 0.4992327632141159
Precision:          0.5187927549923658
Recall:             0.5056146352346248


## Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

# Create and train the Random Forest model
gradientboosting = GradientBoostingClassifier(random_state=2)
# gradientboosting = Grad
# ntBoostingClassifier(random_state=2)

gradientboosting.fit(X_train, y_train)

# Predict the variety for the test set
y_pred = gradientboosting.predict(X_test)

# Decode the predicted variety
predicted_variety = le.inverse_transform(y_pred)

# Print the predicted variety
print(predicted_variety)

# Print the accuracy scores
print(f"{'Training accuracy: ':<20}{gradientboosting.score(X_train, y_train)}")
print(f"{'Testing accuracy: ':<20}{gradientboosting.score(X_test, y_test)}")

print(f"{'F1:':<20}{f1_score(y_test, y_pred, average='weighted')}")
print(f"{'Precision:':<20}{precision_score(y_test, y_pred, average='weighted')}")
print(f"{'Recall:':<20}{recall_score(y_test, y_pred, average='weighted')}")